In [11]:
# run_notears_official_linear.py
# ------------------------------------------------------------
# "DAGs with NO TEARS" (Zheng et al., 2018) 논문 기반 NOTEARS-Linear 구현:
# - Acyclicity: h(W) = tr(exp(W ∘ W)) - d
# - grad h(W) = (exp(W ∘ W)^T) ∘ (2W)
# - Augmented Lagrangian (Algorithm 1) 스타일의 dual ascent
# - 내부 최적화: L-BFGS-B (bound-constrained) + L1을 위한 (W+ - W-) reparam
# - Thresholding + (안전장치) cycle-break
#
# Requirements:
#   pip install numpy pandas scipy networkx
# ------------------------------------------------------------

import os
import json
import warnings
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
import networkx as nx

from scipy.optimize import minimize
from scipy.linalg import expm

warnings.filterwarnings("ignore")


# =========================
# Config
# =========================
DATA_PATH = "./training_data_normalized.csv"   # 사용자 최신 파일명
OUT_DIR = "./dag_out/NOTEARS"

RANDOM_STATE = 42
MAX_FEATURES = None

TARGET_CANDIDATES = ["label", "target", "y", "failure", "bank_failure", "default", "is_failed"]

# NOTEARS hyperparams
LAMBDA1 = 0.1            # sparsity (논문: λ)
LOSS_TYPE = "l2"         # "l2" (연속형 추천)
MAX_OUTER_ITERS = 20     # 논문에서는 보통 ~10 스텝이면 충분하다고 언급(실험적으로)
INNER_MAXITER = 300      # L-BFGS-B 내부 반복
H_TOL = 1e-8             # ε (논문 예: 1e-8 근처)
RHO_INIT = 1.0
RHO_MAX = 1e16
PROGRESS_RATE_C = 0.25   # c in (0,1): "h(W_{t+1}) < c*h(W_t)" 유사하게 쓰기 위한 기준
W_THRESHOLD = 0.0        # ω (논문: threshold). 0.05~0.3 정도로 실험 권장


# =========================
# Data loading
# =========================
def load_numeric_X(
    data_path: str,
    drop_target_candidates: bool = True,
    max_features: Optional[int] = None,
    random_state: int = 42
) -> Tuple[pd.DataFrame, np.ndarray, List[str]]:
    np.random.seed(random_state)
    df = pd.read_csv(data_path, low_memory=False)

    # 흔한 인덱스 컬럼 제거(있으면)
    for c in ["Unnamed: 0", "index", "__index_level_0__"]:
        if c in df.columns:
            df = df.drop(columns=[c])

    if drop_target_candidates:
        # 대소문자/공백 변형 흡수
        norm_cols = {c.strip().lower(): c for c in df.columns}
        drop_cols = []
        for tc in TARGET_CANDIDATES:
            if tc in norm_cols:
                drop_cols.append(norm_cols[tc])
        if drop_cols:
            print(f"[INFO] drop target candidates: {drop_cols}")
            df = df.drop(columns=drop_cols)

    # numeric only
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    df = df[num_cols].copy()

    if max_features is not None and df.shape[1] > max_features:
        df = df.iloc[:, :max_features].copy()
        print(f"[INFO] feature capped: {max_features}")

    # NaN -> median
    for c in df.columns:
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].median())

    col_names = df.columns.tolist()
    X = df.values.astype(float)

    # NOTEARS(LS loss)에서는 mean-center를 기본으로 두는 게 일반적
    X = X - X.mean(axis=0, keepdims=True)

    print(f"[INFO] X shape: {X.shape}")
    return df, X, col_names


# =========================
# NOTEARS core (paper formulas)
# =========================
def loss_l2(W: np.ndarray, X: np.ndarray) -> Tuple[float, np.ndarray]:
    """
    LS loss: ℓ(W;X) = (1/2n) ||X - XW||_F^2
    grad: -(1/n) X^T (X - XW)
    """
    n = X.shape[0]
    R = X - X @ W
    loss = 0.5 / n * np.sum(R * R)
    grad = -(1.0 / n) * (X.T @ R)
    return loss, grad


def acyclicity_h(W: np.ndarray) -> Tuple[float, np.ndarray]:
    """
    h(W) = tr(exp(W∘W)) - d
    ∇h(W) = (exp(W∘W)^T) ∘ (2W)
    """
    d = W.shape[0]
    M = W * W
    E = expm(M)
    h = np.trace(E) - d
    G_h = (E.T) * (2.0 * W)
    return float(h), G_h


def notears_linear_official(
    X: np.ndarray,
    lambda1: float = 0.1,
    loss_type: str = "l2",
    max_outer_iters: int = 20,
    inner_maxiter: int = 300,
    h_tol: float = 1e-8,
    rho_init: float = 1.0,
    rho_max: float = 1e16,
    progress_rate_c: float = 0.25,
    w_threshold: float = 0.0,
    random_state: int = 42
) -> np.ndarray:
    """
    Augmented Lagrangian 스타일 (논문 Algorithm 1의 핵심 아이디어):
      - W_{t+1} = argmin_W  L_rho(W, alpha_t)
      - alpha_{t+1} = alpha_t + rho * h(W_{t+1})
      - h(W_{t+1}) < eps 이면 종료
      - (실전) rho를 점진적으로 키워 h를 억제

    L_rho(W, alpha) = ℓ(W;X) + lambda1 ||W||_1 + (rho/2) h(W)^2 + alpha h(W)

    L1 처리를 위해 W = W+ - W-, W+,W- >= 0 로 재파라미터화하여
    ||W||_1 = sum(W+ + W-) 로 표현합니다.
    """
    np.random.seed(random_state)
    n, d = X.shape

    if loss_type != "l2":
        raise ValueError("This script implements official linear NOTEARS with l2 loss (continuous data).")

    # v = [vec(W+), vec(W-)]
    v = np.zeros(2 * d * d, dtype=float)

    # bounds: diagonal fixed to 0 for both W+ and W-
    bnds = []
    for i in range(d):
        for j in range(d):
            if i == j:
                bnds.append((0.0, 0.0))
            else:
                bnds.append((0.0, None))
    bnds = bnds * 2

    def vec_to_W(vv: np.ndarray) -> np.ndarray:
        Wp = vv[: d * d].reshape(d, d)
        Wm = vv[d * d :].reshape(d, d)
        return Wp - Wm

    def objective_and_grad(vv: np.ndarray, rho: float, alpha: float) -> Tuple[float, np.ndarray]:
        W = vec_to_W(vv)

        # smooth loss
        loss, grad_loss = loss_l2(W, X)

        # acyclicity
        h, grad_h = acyclicity_h(W)

        # objective
        # L1 term: lambda1 * sum(W+ + W-) = lambda1 * sum(vv)
        obj = loss + lambda1 * np.sum(vv) + 0.5 * rho * (h * h) + alpha * h

        # gradient wrt W
        G_W = grad_loss + (rho * h + alpha) * grad_h

        # map to v-grad: dW/dWp = +1, dW/dWm = -1
        grad_v = np.concatenate([G_W.reshape(-1), (-G_W).reshape(-1)])
        # plus grad of lambda1*sum(vv) => +lambda1 for all entries
        grad_v = grad_v + lambda1

        return float(obj), grad_v

    rho = rho_init
    alpha = 0.0
    h_prev = np.inf

    for t in range(max_outer_iters):
        sol = minimize(
            fun=lambda vv: objective_and_grad(vv, rho, alpha)[0],
            x0=v,
            jac=lambda vv: objective_and_grad(vv, rho, alpha)[1],
            method="L-BFGS-B",
            bounds=bnds,
            options={"maxiter": inner_maxiter, "ftol": 1e-12}
        )

        v = sol.x
        W = vec_to_W(v)

        # enforce diag = 0
        np.fill_diagonal(W, 0.0)

        h_val, _ = acyclicity_h(W)

        print(f"[NOTEARS] outer={t+1:03d}  obj={sol.fun:.6f}  h={h_val:.3e}  rho={rho:.1e}  alpha={alpha:.3e}")

        # stopping
        if h_val <= h_tol or rho >= rho_max:
            break

        # rho update (논문 Algorithm 1의 "h가 충분히 줄어들 때까지 rho 선택"의 실전형)
        # 목표: h(W_{t+1}) < c * h(W_t) 유사
        if np.isfinite(h_prev) and h_val < progress_rate_c * h_prev:
            # progress ok: keep rho
            pass
        else:
            # progress 부족: rho 증가
            rho *= 10.0

        # dual ascent
        alpha += rho * h_val
        h_prev = h_val

    # thresholding (논문 4.3)
    if w_threshold and w_threshold > 0:
        W[np.abs(W) < w_threshold] = 0.0

    np.fill_diagonal(W, 0.0)
    return W


# =========================
# Cycle-break (safety DAG enforcement)
# =========================
def break_cycles_by_removing_small_edges(W: np.ndarray) -> np.ndarray:
    """
    그래프에 cycle이 남아있으면, cycle에 포함된 edge 중 |weight|가 가장 작은 edge를 제거 반복.
    """
    W2 = W.copy()
    d = W2.shape[0]

    def build_adj():
        G = {i: [] for i in range(d)}
        for i in range(d):
            for j in range(d):
                if i != j and abs(W2[i, j]) > 0:
                    G[i].append(j)
        return G

    def find_cycle_edges(G):
        color = [0] * d
        parent = [-1] * d

        def dfs(u):
            color[u] = 1
            for v in G[u]:
                if color[v] == 0:
                    parent[v] = u
                    cyc = dfs(v)
                    if cyc is not None:
                        return cyc
                elif color[v] == 1:
                    # back-edge => cycle
                    nodes = [v]
                    cur = u
                    while cur != v and cur != -1:
                        nodes.append(cur)
                        cur = parent[cur]
                    nodes.append(v)
                    nodes = nodes[::-1]
                    return [(a, b) for a, b in zip(nodes[:-1], nodes[1:])]
            color[u] = 2
            return None

        for s in range(d):
            if color[s] == 0:
                cyc = dfs(s)
                if cyc is not None:
                    return cyc
        return None

    removed = 0
    while True:
        cyc = find_cycle_edges(build_adj())
        if cyc is None:
            break
        mags = [(abs(W2[i, j]), i, j) for (i, j) in cyc]
        mags.sort(key=lambda x: x[0])
        _, i_min, j_min = mags[0]
        W2[i_min, j_min] = 0.0
        removed += 1

    if removed:
        print(f"[INFO] cycle-break removed edges: {removed}")

    return W2


# =========================
# Save artifacts
# =========================
def save_artifacts(W: np.ndarray, col_names: List[str], out_dir: str, alg_name: str) -> None:
    os.makedirs(out_dir, exist_ok=True)

    edges = []
    for i, src in enumerate(col_names):
        for j, tgt in enumerate(col_names):
            if i != j and abs(W[i, j]) > 0:
                edges.append([src, tgt, float(W[i, j])])

    edge_df = pd.DataFrame(edges, columns=["source", "target", "weight"])
    edge_path = os.path.join(out_dir, f"edges_{alg_name}.csv")
    edge_df.to_csv(edge_path, index=False)

    adj_df = pd.DataFrame(W, index=col_names, columns=col_names)
    adj_path = os.path.join(out_dir, f"adj_{alg_name}.csv")
    adj_df.to_csv(adj_path)

    G = nx.DiGraph()
    for n in col_names:
        G.add_node(n)
    for _, r in edge_df.iterrows():
        G.add_edge(r["source"], r["target"], weight=float(r["weight"]))

    graphml_path = os.path.join(out_dir, f"graph_{alg_name}.graphml")
    gexf_path = os.path.join(out_dir, f"graph_{alg_name}.gexf")
    nx.write_graphml(G, graphml_path)
    nx.write_gexf(G, gexf_path)

    nodes = [{"id": n} for n in G.nodes()]
    jedges = [{"source": u, "target": v, "weight": float(G[u][v].get("weight", 0.0))} for u, v in G.edges()]
    json_path = os.path.join(out_dir, f"graph_{alg_name}.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({"nodes": nodes, "edges": jedges}, f, ensure_ascii=False, indent=2)

    print(f"[SAVE] {alg_name}")
    print(f"  - {edge_path} (n_edges={len(edge_df)})")
    print(f"  - {adj_path}")
    print(f"  - {graphml_path}")
    print(f"  - {gexf_path}")
    print(f"  - {json_path}")


# =========================
# Main
# =========================
def main():
    _, X, col_names = load_numeric_X(
        DATA_PATH,
        drop_target_candidates=True,
        max_features=MAX_FEATURES,
        random_state=RANDOM_STATE
    )

    print("[RUN] NOTEARS-linear (paper-based, augmented Lagrangian)")
    W = notears_linear_official(
        X,
        lambda1=LAMBDA1,
        loss_type=LOSS_TYPE,
        max_outer_iters=MAX_OUTER_ITERS,
        inner_maxiter=INNER_MAXITER,
        h_tol=H_TOL,
        rho_init=RHO_INIT,
        rho_max=RHO_MAX,
        progress_rate_c=PROGRESS_RATE_C,
        w_threshold=W_THRESHOLD,
        random_state=RANDOM_STATE
    )

    # Safety DAG enforcement (threshold 후 cycle이 남는 경우 대비)
    W = break_cycles_by_removing_small_edges(W)

    save_artifacts(W, col_names, OUT_DIR, "NOTEARS")
    print("[DONE] NOTEARS artifacts saved.")


if __name__ == "__main__":
    main()


[INFO] drop target candidates: ['label']
[INFO] X shape: (17881, 13)
[RUN] NOTEARS-linear (paper-based, augmented Lagrangian)
[NOTEARS] outer=001  obj=4.143689  h=3.200e-01  rho=1.0e+00  alpha=0.000e+00
[NOTEARS] outer=002  obj=4.545449  h=4.070e-02  rho=1.0e+01  alpha=3.200e+00
[NOTEARS] outer=003  obj=4.561189  h=3.677e-02  rho=1.0e+01  alpha=3.607e+00
[NOTEARS] outer=004  obj=4.670966  h=1.582e-02  rho=1.0e+02  alpha=7.284e+00
[NOTEARS] outer=005  obj=4.808441  h=3.374e-03  rho=1.0e+03  alpha=2.310e+01
[NOTEARS] outer=006  obj=4.818899  h=2.846e-03  rho=1.0e+03  alpha=2.647e+01
[NOTEARS] outer=007  obj=4.869688  h=8.069e-04  rho=1.0e+04  alpha=5.493e+01
[NOTEARS] outer=008  obj=4.907110  h=1.989e-04  rho=1.0e+05  alpha=1.356e+02
[NOTEARS] outer=009  obj=4.910755  h=1.688e-04  rho=1.0e+05  alpha=1.555e+02
[NOTEARS] outer=010  obj=4.928732  h=4.770e-05  rho=1.0e+06  alpha=3.243e+02
[NOTEARS] outer=011  obj=4.941337  h=1.088e-05  rho=1.0e+07  alpha=8.013e+02
[NOTEARS] outer=012  obj=4.